# HAM10000 — Stage 1: Data Loading & Exploration

This notebook covers:
- Cloning the repo and adding `src/` to the Python path
- Downloading the HAM10000 dataset via the Kaggle API
- Loading metadata and verifying data integrity
- Visualising class distribution (counts + percentages)
- Displaying sample images per class
- Quantifying the imbalance problem explicitly

> **Run environment:** Google Colab (free tier, T4 GPU).  
> All code that defines functions lives in `src/` — this notebook *calls* those functions.

## Cell 1 — Clone the repo & set up the Python path

We clone the GitHub repo so Colab has the `src/` package available for import.  
We then insert `src/` at position 0 on `sys.path` so `import data_loader` resolves to our module, not any installed package with the same name.

**Why `sys.path.insert(0, ...)`?**  
Position 0 means Python checks our `src/` directory *first*, before site-packages. This avoids name collisions and makes the import order explicit and reproducible.

In [ ]:
import subprocess, sys, os

REPO_URL  = "https://github.com/Dev252001/HAM10000.git"
REPO_DIR  = "/content/ham10000-classifier"
SRC_DIR   = os.path.join(REPO_DIR, "src")

# Clone if not already present (idempotent: re-running the cell is safe)
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")

# Add src/ to Python path so we can do `from data_loader import ...`
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"src/ on path: {SRC_DIR}")

## Cell 2 — Install dependencies

Colab pre-installs most of these, but we pin versions from `requirements.txt`  
so the notebook is reproducible outside Colab too.

In [ ]:
!pip install -q -r /content/ham10000-classifier/requirements.txt

## Cell 3 — Kaggle credentials + dataset download

Upload your `kaggle.json` when prompted.  
The `download_dataset()` function in `data_loader.py` is **idempotent**: if the dataset already exists it skips the download, so re-running is safe.

In [ ]:
from google.colab import files as colab_files
import os, shutil

# Upload kaggle.json
colab_files.upload()  # select your kaggle.json file

# Move it to the location the Kaggle CLI expects
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials configured.")

# Download dataset using the function from src/data_loader.py
from data_loader import download_dataset

DATA_DIR = "/content/ham10000-classifier/data"
download_dataset(DATA_DIR)

## Cell 4 — Load metadata & verify integrity

`load_metadata()` reads the CSV, scans both image folders, and joins them.  
It raises a `ValueError` immediately if any image is missing from disk —  
better to fail loudly here than silently corrupt the dataset later.

In [ ]:
from data_loader import load_metadata, CLASSES, LABEL_MAP, MALIGNANT_CLASSES, CLASS_TO_IDX

df = load_metadata(DATA_DIR)

print(f"Total rows            : {len(df)}")
print(f"Columns               : {list(df.columns)}")
print(f"Unique classes        : {df['dx'].nunique()}")
print(f"Any missing filepaths : {df['filepath'].isna().sum()}")
print()
df.head(3)

## Cell 5 — Class distribution table

We build the distribution table here in the notebook (not in `data_loader.py`)  
because it is purely exploratory/display logic — it doesn't belong in a module  
that will be imported during training.

In [ ]:
import pandas as pd
import numpy as np

counts      = df["dx"].value_counts().reindex(CLASSES)
percentages = (counts / len(df) * 100).round(2)

distribution = pd.DataFrame({
    "Code"        : CLASSES,
    "Full Name"   : [LABEL_MAP[c] for c in CLASSES],
    "Count"       : counts.values,
    "Percentage %": percentages.values,
    "Malignant"   : ["YES" if c in MALIGNANT_CLASSES else "no" for c in CLASSES],
})

print("Class Distribution:")
print(distribution.to_string(index=False))

majority_count = counts.max()
minority_count = counts.min()
print(f"\nImbalance ratio (majority / minority): {majority_count / minority_count:.1f}×")

## Cell 6 — Bar chart: class distribution

Red = malignant, blue = benign. The colour coding immediately communicates  
that the clinically dangerous classes are also the minority classes.

In [ ]:
import matplotlib.pyplot as plt
import os

FIGURES_DIR = "/content/ham10000-classifier/outputs/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

bar_colors = ["#c0392b" if c in MALIGNANT_CLASSES else "#2980b9" for c in CLASSES]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(CLASSES, counts.values, color=bar_colors, edgecolor="white", linewidth=0.8)

for bar, count, pct in zip(bars, counts.values, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{count}\n({pct}%)",
        ha="center", va="bottom", fontsize=9
    )

ax.set_xlabel("Diagnosis Code", fontsize=11)
ax.set_ylabel("Number of Images", fontsize=11)
ax.set_title("HAM10000 Class Distribution\n(red = malignant, blue = benign)", fontsize=13)
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels([f"{c}\n{LABEL_MAP[c]}" for c in CLASSES], fontsize=8)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, "class_distribution.png")
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Saved → {save_path}")

## Cell 7 — Sample images: 5 per class

We visually inspect what the model will learn from.  
Fixed seed (42) → same image grid every time this cell runs, so the saved figure is stable.

In [ ]:
import matplotlib.image as mpimg

N_SAMPLES  = 5
n_classes  = len(CLASSES)
rng        = np.random.default_rng(seed=42)

fig, axes = plt.subplots(n_classes, N_SAMPLES, figsize=(N_SAMPLES * 2.5, n_classes * 2.5))
fig.suptitle("Sample Images per Class  (⚠ = malignant)", fontsize=14, y=1.01)

for row_idx, cls in enumerate(CLASSES):
    class_df   = df[df["dx"] == cls]
    sample_rows = class_df.sample(
        n=min(N_SAMPLES, len(class_df)),
        random_state=int(rng.integers(1000))
    )

    for col_idx, (_, row) in enumerate(sample_rows.iterrows()):
        ax  = axes[row_idx, col_idx]
        img = mpimg.imread(row["filepath"])
        ax.imshow(img)
        ax.axis("off")

        if col_idx == 0:
            flag = " ⚠" if cls in MALIGNANT_CLASSES else ""
            ax.set_ylabel(
                f"{cls}\n{LABEL_MAP[cls]}{flag}",
                fontsize=8, rotation=0, labelpad=65, va="center"
            )

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, "sample_images_per_class.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

## Cell 8 — Imbalance summary (numbers for your writeup)

"Heavily imbalanced" is not a number. These are.

In [ ]:
total    = len(df)
nv_count = counts["nv"]

print("=" * 60)
print("IMBALANCE SUMMARY")
print("=" * 60)
print(f"\nTotal images            : {total}")
print(f"Dominant class (nv)     : {nv_count} images = {nv_count/total*100:.1f}% of dataset")

print("\nMalignant class counts and share:")
for cls in ("mel", "bcc", "akiec"):
    c = counts[cls]
    print(f"  {cls:6s} ({LABEL_MAP[cls]:30s}): {c:5d} images = {c/total*100:.1f}%")

print(f"\nImbalance ratio (nv : vasc) = {counts['nv']} : {counts['vasc']} "
      f"= {counts['nv'] / counts['vasc']:.0f}:1")

print("""
WHY THIS MATTERS FOR MODELLING:
  - A model that always predicts 'nv' gets ~67% accuracy.
  - Accuracy is therefore misleading — it looks good while
    completely failing to detect melanoma or BCC.
  - We will track recall on malignant classes (mel, bcc, akiec)
    and macro-F1 as our primary metrics throughout this project.
  - The imbalance must be handled explicitly in Stage 2/3:
    options are class-weighted loss, oversampling, or focal loss.
""")

---
## Stage 1 complete ✓

**Before moving to Stage 2, verify:**
- [ ] `Total rows: 10015`, `Unique classes: 7`, `Any missing filepaths: 0`
- [ ] Bar chart shows nv towering over all others (~6705 images, ~66.9%)
- [ ] Imbalance ratio prints as ~42:1
- [ ] Sample grid shows all 7 classes with 5 images each
- [ ] Both figures saved to `outputs/figures/`

Next: **Stage 2 — Preprocessing + Split** (stratified split, normalization, augmentation strategy).